# FLUX ARC-AGI-3 Live Test — Full Cognitive System

**Testing FLUX on REAL ARC-AGI-3 games** with the complete cognitive stack:

| Component | Purpose |
|-----------|--------|
| **SpatialMemory** | Ice (curiosity) + Water (exploration mass) navigation |
| **CausalTracker** | Action → effect learning |
| **RuleInducer** | Pattern → rule abstraction |
| **GoalPlanner** | Objective decomposition |
| **PerceptionField** | Active vision with fovea |
| **GridToWave** | Grid → 432D wave encoding |

This notebook connects to the **real ARC-AGI-3 API** to test FLUX on games like:
- `ls20` — Agent reasoning
- `ft09` — Elementary logic
- `vc33` — Orchestration

---
## Cell 1: Environment Setup

In [ ]:
%%time
import os
import sys
from pathlib import Path

# Detect environment
IN_KAGGLE = os.path.exists('/kaggle')
IN_COLAB = 'google.colab' in sys.modules

if IN_KAGGLE:
    ROOT = Path('/kaggle/working/FLUX')
    if not ROOT.exists():
        !git clone https://github.com/Unseengap/FLUX.git {ROOT}
    os.chdir(ROOT)
elif IN_COLAB:
    ROOT = Path('/content/FLUX')
    if not ROOT.exists():
        !git clone https://github.com/Unseengap/FLUX.git {ROOT}
    os.chdir(ROOT)
else:
    ROOT = Path('.').resolve()
    while ROOT.name and not (ROOT / 'flux_utils.py').exists():
        ROOT = ROOT.parent
    if not (ROOT / 'flux_utils.py').exists():
        ROOT = Path('/Users/admin/Desktop/flux')

print(f"Environment: {'Kaggle' if IN_KAGGLE else 'Colab' if IN_COLAB else 'Local'}")
print(f"Root: {ROOT}")
os.chdir(ROOT)

# Add all phase paths
phase_paths = [
    ROOT,
    ROOT / 'phases/phase1',
    ROOT / 'phases/phase2',
    ROOT / 'phases/phase8',
    ROOT / 'phases/phase8_8',
    ROOT / 'phases/phase9_arc',    # Spatial Memory
    ROOT / 'phases/phase_unified', # Cognitive Layer
]
for p in phase_paths:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

print(f"\u2713 Environment configured with {len(phase_paths)} paths")

## Cell 2: Install Dependencies & Load API Key

In [ ]:
# Install arc-agi toolkit if not present
try:
    import arc_agi
    print("\u2713 arc-agi already installed")
except ImportError:
    print("Installing arc-agi...")
    !pip install -q arc-agi
    import arc_agi
    print("\u2713 arc-agi installed")

# ─────────────────────────────────────────────
# Load API Keys from Secrets (Kaggle/Colab first, then .env fallback)
# ─────────────────────────────────────────────

ARC_API_KEY = None
HF_TOKEN = None

# 1. Try Kaggle secrets first
if IN_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        ARC_API_KEY = secrets.get_secret('ARC_API_KEY')
        HF_TOKEN = secrets.get_secret('HF_TOKEN')
        if ARC_API_KEY:
            print("\u2713 ARC_API_KEY loaded from Kaggle secrets")
        if HF_TOKEN:
            print("\u2713 HF_TOKEN loaded from Kaggle secrets")
    except Exception as e:
        print(f"\u26a0 Kaggle secrets error: {e}")

# 2. Try Colab secrets
elif IN_COLAB:
    try:
        from google.colab import userdata
        ARC_API_KEY = userdata.get('ARC_API_KEY')
        HF_TOKEN = userdata.get('HF_TOKEN')
        if ARC_API_KEY:
            print("\u2713 ARC_API_KEY loaded from Colab secrets")
        if HF_TOKEN:
            print("\u2713 HF_TOKEN loaded from Colab secrets")
    except Exception as e:
        print(f"\u26a0 Colab secrets error: {e}")

# 3. Try environment variables
if not ARC_API_KEY:
    ARC_API_KEY = os.environ.get('ARC_API_KEY')
    if ARC_API_KEY:
        print("\u2713 ARC_API_KEY loaded from environment")

if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN')
    if HF_TOKEN:
        print("\u2713 HF_TOKEN loaded from environment")

# 4. Fallback to .env file (local development only)
if not ARC_API_KEY or not HF_TOKEN:
    env_file = ROOT / '.env'
    if env_file.exists():
        print("Loading from .env (local fallback)...")
        with open(env_file) as f:
            for line in f:
                line = line.strip()
                if not ARC_API_KEY and line.startswith('ARC_API_KEY='):
                    ARC_API_KEY = line.split('=', 1)[1]
                    print("\u2713 ARC_API_KEY loaded from .env")
                if not HF_TOKEN and line.startswith('hf_token='):
                    HF_TOKEN = line.split('=', 1)[1]
                    print("\u2713 HF_TOKEN loaded from .env")

# Set environment variables for libraries that need them
if ARC_API_KEY:
    os.environ['ARC_API_KEY'] = ARC_API_KEY
    print(f"  ARC_API_KEY: {ARC_API_KEY[:8]}...{ARC_API_KEY[-4:]}")
else:
    print("\u2717 ARC_API_KEY not found!")
    print("  Add to Kaggle/Colab secrets or set as environment variable")

if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    print(f"  HF_TOKEN: {HF_TOKEN[:8]}...{HF_TOKEN[-4:]}")
else:
    print("\u26a0 HF_TOKEN not found (model download may fail for private repos)")

## Cell 3: Device Setup & Load Model

In [ ]:
%%time
import torch
import numpy as np
from datetime import datetime

# Device
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f"Device: {device}")
if device == 'cuda':
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Load Flux-ARC.flx or download from HuggingFace
CHECKPOINTS_DIR = ROOT / 'checkpoints'
CHECKPOINTS_DIR.mkdir(exist_ok=True)

model_path = CHECKPOINTS_DIR / 'Flux-ARC.flx'

if not model_path.exists():
    print("Downloading Flux-ARC.flx from HuggingFace...")
    from huggingface_hub import hf_hub_download
    
    try:
        hf_hub_download(
            repo_id='UnseenGAP/FLUX',
            filename='checkpoints/Flux-ARC.flx',
            local_dir=str(ROOT),
            token=HF_TOKEN,  # Uses token from secrets loaded in Cell 2
        )
        print("\u2713 Downloaded Flux-ARC.flx")
    except Exception as e:
        print(f"\u26a0 Flux-ARC.flx download failed: {e}")
        print("  Trying Flux-Base.flx fallback...")
        try:
            hf_hub_download(
                repo_id='UnseenGAP/FLUX',
                filename='checkpoints/Flux-Base.flx',
                local_dir=str(ROOT),
                token=HF_TOKEN,
            )
            model_path = CHECKPOINTS_DIR / 'Flux-Base.flx'
            print("\u2713 Downloaded Flux-Base.flx")
        except Exception as e2:
            print(f"\u2717 All downloads failed: {e2}")
            raise

# Load model
print(f"\nLoading {model_path.name}...")
flux_model = torch.load(str(model_path), map_location='cpu', weights_only=False)

print(f"\u2713 Loaded FLUX model")
print(f"  Format: {flux_model.get('format', 'unknown')}")
print(f"  Version: {flux_model.get('version', 'unknown')}")
print(f"  Components: {list(flux_model.get('components', {}).keys())[:8]}...")

## Cell 4: Initialize Cognitive Layer

In [ ]:
%%time
print("Initializing Cognitive Layer...")
print("=" * 60)

# Import cognitive components
from causal_tracker import CausalTracker
from rule_inducer import RuleInducer
from goal_planner import GoalPlanner, GoalType
from perception_field import PerceptionField
from arc_interface import GameAction, GameState, ActionCommand

# Initialize from model or fresh
if 'cognitive' in flux_model:
    print("Loading cognitive layer from checkpoint...")
    cog = flux_model['cognitive']
    
    ct_config = cog['causal_tracker']['config']
    causal_tracker = CausalTracker(
        max_history=ct_config.get('max_history', 1000),
        device='cpu',
    )
    causal_tracker.load_state_dict(cog['causal_tracker']['state_dict'])
    
    ri_config = cog['rule_inducer']['config']
    rule_inducer = RuleInducer(
        causal_tracker=causal_tracker,
        min_observations=ri_config.get('min_observations', 2),
        min_confidence=ri_config.get('min_confidence', 0.5),
    )
    rule_inducer.load_state_dict(cog['rule_inducer']['state_dict'])
    
    goal_planner = GoalPlanner()
    goal_planner.load_state_dict(cog['goal_planner']['state_dict'])
    
    pf_config = cog['perception_field']['config']
    perception_field = PerceptionField(
        max_size=pf_config.get('max_size', 64),
        fovea_radius=pf_config.get('fovea_radius', 5),
    )
    perception_field.load_state_dict(cog['perception_field']['state_dict'])
    
    print("\u2713 Loaded cognitive layer from Flux-ARC.flx")
else:
    print("Creating fresh cognitive layer...")
    causal_tracker = CausalTracker(max_history=1000, device='cpu')
    rule_inducer = RuleInducer(causal_tracker=causal_tracker)
    goal_planner = GoalPlanner()
    perception_field = PerceptionField(max_size=64, fovea_radius=5)
    print("\u2713 Created fresh cognitive layer")

print(f"\nCognitive Components Ready:")
print(f"  \u2713 CausalTracker (links: {len(causal_tracker.causal_links)})")
print(f"  \u2713 RuleInducer (rules: {len(rule_inducer.rules)})")
print(f"  \u2713 GoalPlanner")
print(f"  \u2713 PerceptionField (fovea_r={perception_field.fovea_radius})")

## Cell 5: Initialize Spatial Memory (Ice & Water)

In [ ]:
%%time
print("Initializing Spatial Memory (Ice & Water)...")
print("=" * 60)

from spatial_memory import SpatialMemory, NavigationTarget
import torch
import torch.nn.functional as F

# Create spatial memory instance
spatial_memory = SpatialMemory(
    max_size=64,          # ARC grids up to 64x64
    feature_dim=64,
    curiosity_threshold=0.1,
    device='cpu',
)

# ─────────────────────────────────────────────
# Monkey-patch encode_cell to handle colors >= 10
# ARC-AGI-3 may use extended color palette
# ─────────────────────────────────────────────
_original_encode_cell = spatial_memory.encode_cell

def _safe_encode_cell(self, color: int):
    """Safe encode_cell that clamps colors to 0-9."""
    safe_color = max(0, min(9, color))  # Clamp to valid range
    one_hot = F.one_hot(torch.tensor(safe_color), num_classes=10).float()
    one_hot = one_hot.to(self.device)
    return self.cell_encoder(one_hot)

# Bind the method
import types
spatial_memory.encode_cell = types.MethodType(_safe_encode_cell, spatial_memory)

print("✓ SpatialMemory initialized")
print("✓ Patched encode_cell to handle colors >= 10 (clamped to 0-9)")
print(f"  Max grid size: 64x64")
print(f"  Feature dim: 64")
print(f"  Curiosity threshold: 0.1")
print()
print("Dual-Field System:")
print("  ❄ CURIOSITY ICE — Highlights interesting locations")
print("  ≈ EXPLORATION MASS — Tracks visited locations")

## Cell 6: Initialize Grid Encoder (GridToWave)

In [ ]:
%%time
print("Initializing GridToWave Encoder...")
print("=" * 60)

from grid_adapters import GridToWave, WaveToGrid

WAVE_DIM = 432

# Initialize encoder with LARGER max_size for ARC-AGI-3 64x64 grids
# and num_colors=16 in case ARC-AGI-3 uses extended color palette
grid_to_wave = GridToWave(
    wave_dim=WAVE_DIM,
    max_size=64,      # ARC-AGI-3 uses 64x64 grids
    num_colors=16,    # Extended color range (0-15) for safety
    device='cpu'
)

# Try to load trained weights from model
if 'grid_adapters' in flux_model and 'encoder' in flux_model['grid_adapters']:
    try:
        grid_to_wave.load_state_dict(flux_model['grid_adapters']['encoder'], strict=False)
        print("✓ Loaded trained GridToWave from checkpoint (partial match)")
    except Exception as e:
        print(f"⚠ Could not load trained weights: {e}")
        print("  Using fresh GridToWave")
else:
    print("⚠ Using fresh GridToWave (no trained weights)")

grid_to_wave.eval()

# Test encoding with 64x64 grid
test_grid = [[0]*64 for _ in range(64)]
test_grid[32][32] = 5
with torch.no_grad():
    test_wave = grid_to_wave.encode(test_grid, mode='holistic')
    print(f"\nTest encoding: 64x64 grid -> [{WAVE_DIM}] wave")
    print(f"  Wave norm: {test_wave.norm().item():.4f}")

## Cell 7: Define FLUX-ARC Agent

In [ ]:
import torch.nn.functional as F
from collections import deque
import random

class FLUXARCAgent:
    """
    FLUX Agent for ARC-AGI-3 with full cognitive stack:
    - Spatial Memory (Ice & Water navigation)
    - CausalTracker (action → effect learning)
    - RuleInducer (pattern → rule abstraction)
    - GoalPlanner (objective decomposition)
    - PerceptionField (active vision)
    - GridToWave (grid encoding to 432D waves)
    
    v2 Improvements:
    - Movement-based agent detection (tracks position changes)
    - Click exploration (avoids repeating ineffective clicks)
    """
    
    def __init__(self, verbose: bool = True):
        self.verbose = verbose
        self.reset()
        
    def reset(self):
        """Reset agent state for new game."""
        # Reset spatial memory
        spatial_memory.reset()
        
        # Reset perception
        perception_field.reset()
        
        # Agent state
        self.current_pos = (0, 0)
        self.detected_agent_pos = None  # Position detected by movement
        self.current_target = None
        self.last_grid = None
        self.last_wave = None
        self.action_history = []
        self.observations = []
        self.step_count = 0
        
        # Movement tracking for agent detection
        self.last_action_was_move = False
        self.movement_candidates = []  # Cells that changed on movement
        
        # Click exploration tracking
        self.click_history = []  # (x, y) of past clicks
        self.click_no_effect_count = {}  # {(x,y): count} - clicks with no change
        self.explored_click_cells = set()  # Cells we've tried clicking
        
        # Performance tracking
        self.wins = 0
        self.total_actions = 0
        
    def log(self, msg: str):
        """Log message if verbose."""
        if self.verbose:
            print(f"  [FLUX] {msg}")
    
    def normalize_frame(self, frame) -> list:
        """Normalize ARC API frame to standard 2D grid format."""
        if isinstance(frame, dict):
            if 'grid' in frame:
                frame = frame['grid']
            elif 'frame' in frame:
                frame = frame['frame']
        
        if not isinstance(frame, list):
            self.log(f"Warning: frame is {type(frame)}, using fallback")
            return [[0] * 10 for _ in range(10)]
        
        if len(frame) == 0:
            return [[0] * 10 for _ in range(10)]
        
        # Check for triple nesting: [[[...]]]
        if isinstance(frame[0], list) and len(frame[0]) > 0 and isinstance(frame[0][0], list):
            frame = frame[0]
            self.log(f"Unwrapped triple-nested frame -> {len(frame)} rows")
        
        if len(frame) == 0:
            return [[0] * 10 for _ in range(10)]
        
        if isinstance(frame[0], list):
            if len(frame[0]) > 0 and isinstance(frame[0][0], int):
                return frame
            elif len(frame[0]) > 0 and isinstance(frame[0][0], list):
                frame = frame[0]
                return frame if len(frame) > 0 and isinstance(frame[0], list) else [frame]
        
        if isinstance(frame[0], int):
            return [frame]
        
        self.log(f"Warning: Unknown frame structure")
        return [[0] * 10 for _ in range(10)]
    
    def clamp_grid(self, grid: list, max_val: int = 9) -> list:
        """Clamp all grid values to [0, max_val] for safe encoding."""
        return [
            [max(0, min(max_val, val)) if isinstance(val, int) else 0 for val in row]
            for row in grid
        ]
    
    def encode_grid(self, grid: list) -> torch.Tensor:
        """Encode grid to 432D wave using FLUX."""
        if len(grid) == 0 or len(grid[0]) == 0:
            return torch.zeros(432)
        
        clamped = self.clamp_grid(grid, max_val=15)
        
        try:
            with torch.no_grad():
                wave = grid_to_wave.encode(clamped, mode='holistic')
            return wave
        except Exception as e:
            self.log(f"Encoding error: {e}, using zero wave")
            return torch.zeros(432)
    
    def find_agent_position(self, grid: list) -> tuple:
        """
        Find agent position using multiple strategies:
        1. If we detected agent by movement, use that
        2. Look for common agent colors (1, 2, 3, 8, 9)
        3. Look for unique/rare cells (potential agent markers)
        4. Default to grid center
        """
        if not grid or not grid[0]:
            return (0, 0)
        
        grid_h, grid_w = len(grid), len(grid[0])
        
        # Strategy 1: Use movement-detected position
        if self.detected_agent_pos is not None:
            return self.detected_agent_pos
        
        # Strategy 2: Look for common agent colors
        agent_colors = [1, 2, 3, 8, 9]  # Common agent marker colors
        for color in agent_colors:
            for r, row in enumerate(grid):
                for c, val in enumerate(row):
                    if isinstance(val, int) and val == color:
                        # Found a potential agent - remember it
                        return (r, c)
        
        # Strategy 3: Find rare/unique colored cells (potential agent)
        color_counts = {}
        for r, row in enumerate(grid):
            for c, val in enumerate(row):
                if isinstance(val, int):
                    color_counts[val] = color_counts.get(val, 0) + 1
        
        # Find rarest non-zero color
        if color_counts:
            rarest_color = min(
                (c for c in color_counts if c > 0), 
                key=lambda x: color_counts[x],
                default=None
            )
            if rarest_color is not None and color_counts.get(rarest_color, 0) <= 5:
                for r, row in enumerate(grid):
                    for c, val in enumerate(row):
                        if isinstance(val, int) and val == rarest_color:
                            return (r, c)
        
        # Strategy 4: Default to center
        return (grid_h // 2, grid_w // 2)
    
    def detect_agent_by_movement(self, old_grid: list, new_grid: list, action: int):
        """
        Detect agent position by observing what changed after a movement action.
        The agent is likely at cells that changed value after UP/DOWN/LEFT/RIGHT.
        """
        if action not in [1, 2, 3, 4]:  # Only movement actions
            return
        
        if old_grid is None or new_grid is None:
            return
        
        changes = self.detect_changes(old_grid, new_grid)
        
        if len(changes) >= 2:
            # Movement typically causes 2+ changes: old pos cleared, new pos filled
            # The new agent position should have a different value now
            
            # Find cells that went from background to foreground (agent moved TO)
            # and cells that went from foreground to background (agent moved FROM)
            moved_to = []
            moved_from = []
            
            for r, c, old_val, new_val in changes:
                if old_val == 0 and new_val > 0:
                    moved_to.append((r, c, new_val))
                elif old_val > 0 and new_val == 0:
                    moved_from.append((r, c, old_val))
            
            # Agent likely moved TO one of these positions
            if moved_to:
                # Pick the one consistent with movement direction
                dr, dc = {1: (-1, 0), 2: (1, 0), 3: (0, -1), 4: (0, 1)}.get(action, (0, 0))
                
                for r, c, val in moved_to:
                    self.detected_agent_pos = (r, c)
                    self.log(f"Detected agent at ({r}, {c}) by movement")
                    return
    
    def find_target_cells(self, grid: list) -> list:
        """Find potential target cells (non-zero, non-background, non-agent)."""
        targets = []
        if not grid or not grid[0]:
            return targets
        
        agent_pos = self.find_agent_position(grid)
        
        # Count colors to identify background (most common)
        color_counts = {}
        for r, row in enumerate(grid):
            for c, val in enumerate(row):
                if isinstance(val, int):
                    color_counts[val] = color_counts.get(val, 0) + 1
        
        # Background is most common color
        bg_color = max(color_counts, key=color_counts.get) if color_counts else 0
        
        for r, row in enumerate(grid):
            for c, val in enumerate(row):
                if isinstance(val, int) and val != bg_color and (r, c) != agent_pos:
                    targets.append((r, c, val))
        
        return targets
    
    def detect_changes(self, old_grid: list, new_grid: list) -> list:
        """Detect cell changes between grids."""
        changes = []
        if old_grid is None or not old_grid or not new_grid:
            return changes
        if not old_grid[0] or not new_grid[0]:
            return changes
        for r in range(min(len(old_grid), len(new_grid))):
            for c in range(min(len(old_grid[0]), len(new_grid[0]))):
                old_val = old_grid[r][c] if isinstance(old_grid[r][c], int) else 0
                new_val = new_grid[r][c] if isinstance(new_grid[r][c], int) else 0
                if old_val != new_val:
                    changes.append((r, c, old_val, new_val))
        return changes
    
    def observe(self, frame, available_actions: list = None):
        """Process observation using full cognitive stack."""
        self.step_count += 1
        
        grid = self.normalize_frame(frame)
        grid_h = len(grid)
        grid_w = len(grid[0]) if grid and grid[0] else 0
        grid_size = (grid_h, grid_w)
        
        self.log(f"Step {self.step_count}: Grid {grid_h}x{grid_w}")
        
        # Detect changes before updating
        changes = self.detect_changes(self.last_grid, grid)
        
        # Detect agent by movement if last action was a move
        if self.action_history and self.last_grid:
            last_action = self.action_history[-1]
            if last_action in [1, 2, 3, 4]:
                self.detect_agent_by_movement(self.last_grid, grid, last_action)
            
            # Track click effectiveness
            if last_action == 6 and self.click_history:
                last_click = self.click_history[-1]
                if len(changes) == 0:
                    # Click had no effect
                    key = (last_click['x'], last_click['y'])
                    self.click_no_effect_count[key] = self.click_no_effect_count.get(key, 0) + 1
                else:
                    # Click caused change!
                    self.log(f"Click at {last_click} caused {len(changes)} changes!")
        
        # 1. Encode grid
        current_wave = self.encode_grid(grid)
        
        # 2. Find agent position
        self.current_pos = self.find_agent_position(grid)
        
        # 3. Update spatial memory
        if grid_h <= spatial_memory.max_size and grid_w <= spatial_memory.max_size and grid_h > 0 and grid_w > 0:
            try:
                clamped_grid = self.clamp_grid(grid, max_val=9)
                obs_result = spatial_memory.observe(
                    position=self.current_pos,
                    local_view=clamped_grid,
                    global_grid=clamped_grid,
                )
            except Exception as e:
                self.log(f"Spatial memory error: {e}")
        
        # 4. Find targets and add curiosity
        targets = self.find_target_cells(grid)
        for r, c, color in targets:
            if r < spatial_memory.max_size and c < spatial_memory.max_size:
                spatial_memory.curiosity_field[r, c] += 2.0
        
        # 5. Detect changes → spawn ice and track causality
        for r, c, old_val, new_val in changes:
            self.log(f"Change at ({r},{c}): {old_val} -> {new_val}")
            if r < spatial_memory.max_size and c < spatial_memory.max_size:
                spatial_memory.curiosity_field[r, c] += 5.0
            
            if self.action_history:
                last_action = self.action_history[-1]
                try:
                    causal_tracker.record(
                        position=self.current_pos,
                        action=last_action,
                        grid_before=np.array(self.last_grid) if self.last_grid else np.zeros((1,1)),
                        grid_after=np.array(grid),
                    )
                except Exception as e:
                    self.log(f"Causal tracking error: {e}")
        
        # 6. Update perception field
        try:
            perception_field.focus(self.current_pos, reason="agent")
        except:
            pass
        
        # 7. Decay fields
        spatial_memory.step_decay()
        
        # 8. Wave change detection
        wave_change = 0.0
        if self.last_wave is not None and current_wave.norm() > 0:
            try:
                cos_sim = F.cosine_similarity(
                    current_wave.unsqueeze(0),
                    self.last_wave.unsqueeze(0)
                ).item()
                wave_change = 1.0 - cos_sim
            except:
                pass
        
        # Store for next step
        self.last_grid = [row[:] for row in grid]
        self.last_wave = current_wave
        self.observations.append({
            'step': self.step_count,
            'pos': self.current_pos,
            'wave_change': wave_change,
            'changes': len(changes),
        })
        
        return {
            'position': self.current_pos,
            'wave_change': wave_change,
            'changes': changes,
            'targets': targets,
            'grid_size': grid_size,
        }
    
    def decide_action(self, frame, available_actions: list) -> int:
        """Decide next action using cognitive stack."""
        grid = self.normalize_frame(frame)
        grid_h = len(grid)
        grid_w = len(grid[0]) if grid and grid[0] else 0
        grid_size = (grid_h, grid_w)
        
        move_actions = [a for a in available_actions if a in [1, 2, 3, 4]]
        has_interact = 5 in available_actions
        has_click = 6 in available_actions
        has_undo = 7 in available_actions
        
        # If only click available, use click action
        if has_click and not move_actions and not has_interact:
            return 6
        
        # Check if we need a new target
        needs_new_target = (
            self.current_target is None or
            self.current_pos == self.current_target.position
        )
        
        if needs_new_target and grid_h > 0 and grid_w > 0:
            try:
                self.current_target = spatial_memory.get_navigation_target(
                    self.current_pos,
                    grid_size,
                )
                if self.current_target:
                    self.log(f"New target: {self.current_target.position} ({self.current_target.reason})")
            except Exception as e:
                self.log(f"Navigation target error: {e}")
        
        # If at target and can interact, do it
        if self.current_target and self.current_pos == self.current_target.position:
            if has_interact:
                self.log("At target, using ACTION5 (interact)")
                return 5
        
        # Navigate toward target
        if self.current_target and move_actions:
            try:
                action = spatial_memory.get_next_action(
                    self.current_pos,
                    self.current_target,
                )
                arc_action = action + 1
                if arc_action in move_actions:
                    return arc_action
            except Exception as e:
                self.log(f"Navigation error: {e}")
        
        # Fallback: random available action
        if move_actions:
            return random.choice(move_actions)
        if has_interact:
            return 5
        if has_click:
            return 6
        
        return available_actions[0] if available_actions else 1
    
    def get_unexplored_click_cell(self, grid: list) -> tuple:
        """
        Find an unexplored cell to click using exploration strategy.
        Avoids cells that were already clicked without effect.
        """
        grid_h = len(grid)
        grid_w = len(grid[0]) if grid and grid[0] else 0
        
        if grid_h == 0 or grid_w == 0:
            return (0, 0)
        
        # Get targets first (interesting cells)
        targets = self.find_target_cells(grid)
        
        # Filter out cells we've tried multiple times
        good_targets = []
        for r, c, val in targets:
            key = (c, r)  # Note: (x, y) = (c, r)
            if self.click_no_effect_count.get(key, 0) < 2:
                good_targets.append((r, c, val))
        
        # If we have good targets, pick one we haven't tried recently
        if good_targets:
            # Prefer targets we haven't clicked yet
            for r, c, val in good_targets:
                if (c, r) not in self.explored_click_cells:
                    self.explored_click_cells.add((c, r))
                    return (r, c)
            # Otherwise pick one with lowest fail count
            r, c, _ = min(good_targets, key=lambda t: self.click_no_effect_count.get((t[1], t[0]), 0))
            return (r, c)
        
        # Use curiosity ice field to find interesting unexplored areas
        try:
            ice = spatial_memory.curiosity_field[:grid_h, :grid_w].clone()
            
            # Penalize cells we've already tried
            for (x, y), count in self.click_no_effect_count.items():
                if 0 <= y < grid_h and 0 <= x < grid_w:
                    ice[y, x] -= count * 10
            
            # Find max ice cell
            if ice.max() > -100:
                flat_idx = ice.argmax().item()
                r, c = divmod(flat_idx, grid_w)
                if (c, r) not in self.explored_click_cells or self.click_no_effect_count.get((c, r), 0) < 3:
                    self.explored_click_cells.add((c, r))
                    return (r, c)
        except:
            pass
        
        # Systematic exploration: divide grid into regions, try each
        region_size = max(1, min(grid_h, grid_w) // 4)
        for start_r in range(0, grid_h, region_size):
            for start_c in range(0, grid_w, region_size):
                key = (start_c + region_size // 2, start_r + region_size // 2)
                if key not in self.explored_click_cells:
                    self.explored_click_cells.add(key)
                    r = min(start_r + region_size // 2, grid_h - 1)
                    c = min(start_c + region_size // 2, grid_w - 1)
                    return (r, c)
        
        # Last resort: random unexplored cell
        for _ in range(10):
            r = random.randint(0, grid_h - 1)
            c = random.randint(0, grid_w - 1)
            if (c, r) not in self.explored_click_cells:
                self.explored_click_cells.add((c, r))
                return (r, c)
        
        # Very last resort: center
        return (grid_h // 2, grid_w // 2)
    
    def get_action_with_data(self, frame, available_actions: list) -> tuple:
        """Get action and any required data (x,y for ACTION6)."""
        grid = self.normalize_frame(frame)
        action = self.decide_action(frame, available_actions)
        data = {}
        
        # ACTION6 needs coordinates
        if action == 6:
            grid_h = len(grid)
            grid_w = len(grid[0]) if grid and grid[0] else 0
            
            if grid_h > 0 and grid_w > 0:
                # Use smart exploration to find next click target
                r, c = self.get_unexplored_click_cell(grid)
                data = {'x': c, 'y': r}
                self.log(f"ACTION6 click at ({c}, {r})")
            else:
                data = {'x': 0, 'y': 0}
            
            # Track click history
            self.click_history.append(data.copy())
        
        self.action_history.append(action)
        self.total_actions += 1
        
        return action, data
    
    def get_stats(self) -> dict:
        """Get agent performance stats."""
        grid_size = (64, 64)
        sm_stats = spatial_memory.get_stats(grid_size)
        return {
            'steps': self.step_count,
            'total_actions': self.total_actions,
            'wins': self.wins,
            'causal_links': len(causal_tracker.causal_links),
            'rules_induced': len(rule_inducer.rules),
            'coverage': sm_stats.get('coverage', 0.0),
            'clicks_explored': len(self.explored_click_cells),
            'detected_agent_pos': self.detected_agent_pos,
        }

# Create global agent instance
agent = FLUXARCAgent(verbose=True)
print("✓ FLUXARCAgent v2 created with IMPROVED strategies")
print()
print("Improvements:")
print("  1. Movement-based agent detection")
print("     - Tracks grid changes after UP/DOWN/LEFT/RIGHT")
print("     - Identifies agent position from cell transitions")
print()
print("  2. Smart click exploration")
print("     - Avoids repeating ineffective clicks")
print("     - Explores grid regions systematically")
print("     - Uses curiosity ice field to prioritize")
print()
print("  3. Better target detection")
print("     - Identifies background color (most common)")
print("     - Targets = non-background, non-agent cells")
print()
print("Full cognitive stack:")
print("  - Spatial Memory (Ice & Water navigation)")
print("  - CausalTracker (action → effect learning)")
print("  - RuleInducer (pattern → rule abstraction)")
print("  - GoalPlanner (objective decomposition)")
print("  - PerceptionField (active vision)")
print("  - GridToWave (grid encoding to 432D waves)")

## Cell 8: Test Connection to ARC API

In [ ]:
import requests

print("Testing ARC-AGI-3 API Connection...")
print("=" * 60)

ROOT_URL = "https://three.arcprize.org"

# Create session
session = requests.Session()
session.headers.update({
    "X-API-Key": ARC_API_KEY,
    "Accept": "application/json"
})

# Get games list
response = session.get(f"{ROOT_URL}/api/games")

if response.status_code == 200:
    games = response.json()
    print(f"\u2713 Connected! Found {len(games)} games")
    print(f"\nAvailable games:")
    for game in games[:10]:
        print(f"  - {game['game_id']}: {game.get('title', 'N/A')}")
    if len(games) > 10:
        print(f"  ... and {len(games) - 10} more")
else:
    print(f"\u2717 API Error: {response.status_code}")
    print(response.text)

## Cell 9: Play Single Game with FLUX Agent

In [ ]:
def play_game_with_flux(game_id: str, max_actions: int = 100, verbose: bool = True):
    """
    Play a single ARC-AGI-3 game using FLUX agent.
    Returns result dict including spatial memory snapshot for visualization.
    """
    print(f"\n{'='*60}")
    print(f"Playing: {game_id}")
    print(f"{'='*60}")
    
    # Reset agent (clears spatial memory for fresh game)
    agent.reset()
    agent.verbose = verbose
    
    # Open scorecard
    response = session.post(
        f"{ROOT_URL}/api/scorecard/open",
        json={"tags": ["flux-test", game_id]}
    )
    if response.status_code != 200:
        print(f"✗ Failed to open scorecard: {response.text}")
        return None
    
    card_id = response.json()["card_id"]
    print(f"Scorecard: {card_id}")
    
    # Reset game
    response = session.post(
        f"{ROOT_URL}/api/cmd/RESET",
        json={"game_id": game_id, "card_id": card_id}
    )
    if response.status_code != 200:
        print(f"✗ RESET failed: {response.text}")
        return None
    
    game_data = response.json()
    guid = game_data["guid"]
    state = game_data["state"]
    score = game_data.get("score", 0)
    frame = game_data.get("frame", [[]])
    available_actions = game_data.get("available_actions", [1, 2, 3, 4, 5])
    
    # Debug frame structure in detail
    print(f"Initial state: {state}, Score: {score}")
    print(f"Frame structure analysis:")
    print(f"  type(frame) = {type(frame)}")
    if isinstance(frame, list):
        print(f"  len(frame) = {len(frame)}")
        if len(frame) > 0:
            print(f"  type(frame[0]) = {type(frame[0])}")
            if isinstance(frame[0], list):
                print(f"  len(frame[0]) = {len(frame[0])}")
                if len(frame[0]) > 0:
                    print(f"  type(frame[0][0]) = {type(frame[0][0])}")
                    if isinstance(frame[0][0], list):
                        print(f"  len(frame[0][0]) = {len(frame[0][0])}")
                        if len(frame[0][0]) > 0:
                            print(f"  type(frame[0][0][0]) = {type(frame[0][0][0])}")
                            print(f"  frame[0][0][:5] = {frame[0][0][:5]}")
                    else:
                        print(f"  frame[0][:5] = {frame[0][:5]}")
    print(f"Available actions: {available_actions}")
    
    # Play loop
    actions_taken = 0
    level_wins = 0
    
    while state == "NOT_FINISHED" and actions_taken < max_actions:
        try:
            # Observe
            obs = agent.observe(frame, available_actions)
            
            # Decide action
            action, action_data = agent.get_action_with_data(frame, available_actions)
            action_name = ['RESET', 'UP', 'DOWN', 'LEFT', 'RIGHT', 'INTERACT', 'CLICK', 'UNDO'][action] if action < 8 else f'ACTION{action}'
            
            if verbose and actions_taken % 10 == 0:
                print(f"\nStep {actions_taken}: pos={agent.current_pos}, action={action_name}, grid={obs.get('grid_size', 'N/A')}")
            
        except Exception as e:
            print(f"✗ Agent error at step {actions_taken}: {e}")
            import traceback
            traceback.print_exc()
            break
        
        # Build request
        request_data = {
            "game_id": game_id,
            "card_id": card_id,
            "guid": guid,
        }
        request_data.update(action_data)
        
        # Take action
        action_endpoint = f"ACTION{action}" if action > 0 else "RESET"
        response = session.post(
            f"{ROOT_URL}/api/cmd/{action_endpoint}",
            json=request_data
        )
        
        if response.status_code != 200:
            print(f"✗ Action failed: {response.text}")
            break
        
        game_data = response.json()
        state = game_data["state"]
        new_score = game_data.get("score", score)
        frame = game_data.get("frame", frame)
        available_actions = game_data.get("available_actions", available_actions)
        
        # Track level wins
        if new_score > score:
            print(f"  ↑ Level completed! Score: {score} -> {new_score}")
            level_wins += 1
        
        score = new_score
        actions_taken += 1
    
    # Close scorecard
    try:
        response = session.post(
            f"{ROOT_URL}/api/scorecard/close",
            json={"card_id": card_id}
        )
    except:
        pass
    
    # ─────────────────────────────────────────────
    # CAPTURE SPATIAL MEMORY SNAPSHOT for this game
    # ─────────────────────────────────────────────
    grid_size = 64  # Default for ARC-AGI-3
    if agent.last_grid:
        grid_size = max(len(agent.last_grid), len(agent.last_grid[0]) if agent.last_grid else 64)
    
    spatial_snapshot = {
        'exploration_mass': spatial_memory.exploration_mass[:grid_size, :grid_size].clone().cpu().numpy(),
        'curiosity_field': spatial_memory.curiosity_field[:grid_size, :grid_size].clone().cpu().numpy(),
        'grid_size': grid_size,
        'final_pos': agent.current_pos,
        'detected_agent_pos': agent.detected_agent_pos,
        'clicks_explored': len(agent.explored_click_cells),
    }
    
    # Results
    result = {
        'game_id': game_id,
        'card_id': card_id,
        'final_state': state,
        'final_score': score,
        'actions_taken': actions_taken,
        'level_wins': level_wins,
        'agent_stats': agent.get_stats(),
        'spatial_snapshot': spatial_snapshot,  # NEW: per-game spatial memory
    }
    
    print(f"\n{'-'*40}")
    print(f"Result: {state}")
    print(f"Score: {score}")
    print(f"Actions: {actions_taken}")
    print(f"Levels won: {level_wins}")
    print(f"Scorecard: {ROOT_URL}/scorecards/{card_id}")
    
    return result

print("✓ play_game_with_flux() defined")
print("  Now captures per-game spatial memory snapshots")

## Cell 10: Run FLUX on ls20

In [ ]:
%%time
# Test on ls20 - Agent Reasoning game
result_ls20 = play_game_with_flux("ls20", max_actions=50, verbose=True)

## Cell 11: Run FLUX on ft09

In [ ]:
%%time
# Test on ft09 - Elementary Logic game
result_ft09 = play_game_with_flux("ft09", max_actions=50, verbose=True)

## Cell 12: Run FLUX on vc33

In [ ]:
%%time
# Test on vc33 - Orchestration game
result_vc33 = play_game_with_flux("vc33", max_actions=50, verbose=True)

## Cell 13: Aggregate Results

In [ ]:
print("\n" + "=" * 60)
print("FLUX ARC-AGI-3 TEST RESULTS")
print("=" * 60)

results = [r for r in [result_ls20, result_ft09, result_vc33] if r is not None]

print(f"\nGames tested: {len(results)}")
print(f"\n{'Game':<15} {'State':<15} {'Score':<10} {'Actions':<10} {'Levels':<10}")
print("-" * 60)

total_score = 0
total_actions = 0
total_wins = 0

for r in results:
    print(f"{r['game_id']:<15} {r['final_state']:<15} {r['final_score']:<10} {r['actions_taken']:<10} {r['level_wins']:<10}")
    total_score += r['final_score']
    total_actions += r['actions_taken']
    total_wins += r['level_wins']

print("-" * 60)
print(f"{'TOTAL':<15} {'':<15} {total_score:<10} {total_actions:<10} {total_wins:<10}")

print(f"\n\nAgent Stats:")
stats = agent.get_stats()
for key, val in stats.items():
    print(f"  {key}: {val}")

print(f"\n\nCognitive Layer Stats:")
print(f"  Causal links learned: {len(causal_tracker.causal_links)}")
print(f"  Rules induced: {len(rule_inducer.rules)}")

## Cell 14: Visualize Spatial Memory

In [ ]:
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────
# Visualize Spatial Memory for EACH GAME separately
# ─────────────────────────────────────────────

results = [r for r in [result_ls20, result_ft09, result_vc33] if r is not None]

if not results:
    print("No results to visualize!")
else:
    n_games = len(results)
    
    # Create figure with 2 rows per game (mass + ice)
    fig, axes = plt.subplots(n_games, 2, figsize=(14, 5 * n_games))
    
    # Handle single game case (axes won't be 2D)
    if n_games == 1:
        axes = [axes]
    
    for i, result in enumerate(results):
        game_id = result['game_id']
        snap = result.get('spatial_snapshot', {})
        
        mass = snap.get('exploration_mass')
        ice = snap.get('curiosity_field')
        grid_size = snap.get('grid_size', 64)
        final_pos = snap.get('final_pos', (0, 0))
        detected_pos = snap.get('detected_agent_pos')
        clicks = snap.get('clicks_explored', 0)
        
        if mass is None or ice is None:
            print(f"[{game_id}] No spatial data captured")
            continue
        
        # Exploration Mass (Water)
        im1 = axes[i][0].imshow(mass, cmap='hot', vmin=0)
        axes[i][0].set_title(f'{game_id}: Exploration Mass (Water)\nWhere FLUX explored')
        axes[i][0].set_xlabel(f'Final pos: {final_pos} | Detected: {detected_pos}')
        plt.colorbar(im1, ax=axes[i][0])
        
        # Mark final position
        if final_pos:
            axes[i][0].scatter([final_pos[1]], [final_pos[0]], c='cyan', s=100, marker='*', label='Final')
        
        # Curiosity Ice
        im2 = axes[i][1].imshow(ice, cmap='cool', vmin=0)
        axes[i][1].set_title(f'{game_id}: Curiosity Ice\nWhat was interesting')
        axes[i][1].set_xlabel(f'Score: {result["final_score"]} | Actions: {result["actions_taken"]} | Clicks: {clicks}')
        plt.colorbar(im2, ax=axes[i][1])
    
    plt.tight_layout()
    plt.show()
    
    # ─────────────────────────────────────────────
    # ASCII visualizations for each game
    # ─────────────────────────────────────────────
    print("\n" + "=" * 80)
    print("SPATIAL MEMORY ASCII VIEW — Per Game")
    print("=" * 80)
    
    for result in results:
        game_id = result['game_id']
        snap = result.get('spatial_snapshot', {})
        grid_size = snap.get('grid_size', 64)
        final_pos = snap.get('final_pos', (0, 0))
        mass = snap.get('exploration_mass')
        ice = snap.get('curiosity_field')
        
        print(f"\n{'─' * 60}")
        print(f"Game: {game_id}")
        print(f"Score: {result['final_score']} | Actions: {result['actions_taken']} | Final Position: {final_pos}")
        print(f"{'─' * 60}")
        
        if mass is None:
            print("  (No spatial data)")
            continue
        
        # Simple ASCII visualization
        h, w = mass.shape
        display_h = min(h, 32)  # Limit for display
        display_w = min(w, 64)
        
        print()
        for r in range(display_h):
            row_str = ""
            for c in range(display_w):
                m = mass[r, c]
                i = ice[r, c]
                
                if (r, c) == final_pos:
                    row_str += "@ "
                elif m > 10:
                    row_str += "● "  # Heavy exploration
                elif m > 1:
                    row_str += "· "  # Light exploration
                elif i > 10:
                    row_str += "🧊"  # High curiosity
                elif i > 1:
                    row_str += "❄ "  # Low curiosity
                else:
                    row_str += "  "  # Unexplored
            print(row_str)
        
        if h > display_h or w > display_w:
            print(f"  ... (showing {display_h}x{display_w} of {h}x{w})")
        
        print(f"\nLegend: @ final pos | ● explored | · visited | 🧊 high ice | ❄ ice")
        print(f"Stats: {snap.get('clicks_explored', 0)} cells clicked")

## Cell 15: Summary

In [ ]:
print("=" * 60)
print("FLUX ARC-AGI-3 LIVE TEST COMPLETE")
print("=" * 60)

print(f"""
Model: {model_path.name}
Games tested: {len(results)}
Total levels won: {total_wins}
Total actions: {total_actions}

Cognitive Stack Used:
  \u2713 SpatialMemory (Ice & Water navigation)
  \u2713 CausalTracker ({len(causal_tracker.causal_links)} links learned)
  \u2713 RuleInducer ({len(rule_inducer.rules)} rules)
  \u2713 GoalPlanner
  \u2713 PerceptionField
  \u2713 GridToWave encoder

View scorecards at:
""")

for r in results:
    print(f"  {r['game_id']}: {ROOT_URL}/scorecards/{r['card_id']}")

print(f"""

Next Steps:
  1. Train GridToWave on more ARC patterns
  2. Improve rule induction from causal observations
  3. Test on more games
  4. Submit to competition (OperationMode.COMPETITION)
""")